<h1><center><strong></strong></center></h1>
<h1><center><strong>CME466</strong></center></h1>
<h1><center><strong>Design of an Advanced Digital System</strong></center></h1>
<p><center><strong>Department of Electrical and Computer Engineering</strong></center></p>
<p><center><strong>2025 Winter Term</strong></center></p>
<p><center><strong>Last updated: March 03, 2026</strong></center></p>


<h1><font color=#ad42f5><strong>Introduction to Neural Networks - Part 2</strong></h1>

# Implement NN for Fashion MNIST

https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Check versions
#print(f"PyTorch version: {torch.__version__}\ntorchvision version: {torchvision.__version__}")

# Device configuration (use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define a transformation to convert images to tensors
transform = transforms.ToTensor()

# Download and load the training data
train_data = datasets.FashionMNIST(
    root="data", # where to download data to?
    train=True, # get training data
    download=True, # download data if it doesn't exist on disk
    transform=transform, # images come as PIL format, we want to turn into Torch tensors
)

# Download and load the test data
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform,
)

# Create data loaders for batching and shuffling
batch_size = 64 # start with 64, then try 32,...
train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
# don't necessarily have to shuffle the testing data

# Class names for reference
class_labels = {
    0: 'T-shirt/top', 1: 'Trouser', 2: 'Pullover', 3: 'Dress', 4: 'Coat',
    5: 'Sandal', 6: 'Shirt', 7: 'Sneaker', 8: 'Bag', 9: 'Ankle boot'
}


In [ ]:
# Let's check out what we've created
print(f"Sample size: {len(train_data.data), len(train_data.targets), len(test_data.data), len(test_data.targets)}")
print(f"Length of train dataloader: {len(train_dataloader)} batches of {batch_size}")
print(f"Length of test dataloader: {len(test_dataloader)} batches of {batch_size}")

In [ ]:
# check the classes
class_names = train_data.classes
class_names

In [ ]:
# Plot some images
torch.manual_seed(40)
fig = plt.figure(figsize=(9, 9))
rows, cols = 4, 4
for i in range(1, rows * cols + 1):
    random_idx = torch.randint(0, len(train_data), size=[1]).item()
    img, label = train_data[random_idx]
    fig.add_subplot(rows, cols, i)
    plt.imshow(img.squeeze(), cmap="gray")
    plt.title(class_names[label])
    plt.axis(False);

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten() # Flattens the 28x28 image into a 784-pixel vector
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(784, 512), # play with the numbers of neurons
            nn.ReLU(), # change to nn.Sigmoid()
            nn.Linear(512, 128),
            nn.ReLU(), # change to nn.Sigmoid()
            nn.Linear(128, 10) # Output layer must have 10 neurons for 10 classes
            # nn.Softmax(dim=1) # see how softmax works, change previous to 32
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)


In [ ]:
# Define hyperparameters
learning_rate = 1e-3 # 0.001

# Define the loss function (Cross Entropy Loss for multi-class classification)
loss_fn = nn.CrossEntropyLoss()

# Define the optimizer (Adam is a common choice)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Forward pass
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch*len(X)
            print(f'loss:{loss:>7f} [{current:>5d}/{size:>5d}]')

In [ ]:
def test(dataloader, model, loss_fn):
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= len(dataloader)
    correct /= len(dataloader.dataset)
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


In [ ]:
# Import tqdm for progress bar
from tqdm.auto import tqdm

epochs = 5
for t in tqdm(range(epochs)):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")


In [ ]:
# define functions for plotting
import numpy as np

def plot_image(i, predictions_array, true_label, img):
  true_label, img = true_label[i], img[i]
  plt.grid(False)
  plt.xticks([])
  plt.yticks([])

  plt.imshow(img, cmap=plt.cm.binary)

  predicted_label = torch.argmax(predictions_array).item()

  if predicted_label == true_label:
    color = 'blue'
  else:
    color = 'red'

  plt.xlabel("{} {:2.0f}% ({})".format(class_names[predicted_label],
                                100*torch.max(predictions_array).item(),
                                class_names[true_label]),
                                color=color)

def plot_value_array(i, predictions_array, true_label):
  true_label = true_label[i]
  plt.grid(False)
  plt.xticks(range(10))
  plt.yticks([])
  thisplot = plt.bar(range(10), predictions_array, color="#777777")
  plt.ylim([0, 1])
  predicted_label = np.argmax(predictions_array)

  thisplot[predicted_label].set_color('red')
  thisplot[true_label].set_color('blue')

In [ ]:
import random

# Plot the first X test images, their predicted labels, and (true labels).
# Color correct predictions in blue and incorrect predictions in red.
num_rows = 5
num_cols = 3
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))

# Convert dataloader into a list of batches
all_batches = list(test_dataloader)

# Pick a random batch
X_batch, y_batch = random.choice(all_batches)
X_batch, y_batch = X_batch.to(device), y_batch.to(device)

# Get model predictions
model.eval()  # Ensure model is in eval mode
with torch.no_grad():
    y_pred = model(X_batch)  # Get raw logits
    y_pred_prob = torch.softmax(y_pred, dim=1)  # Convert logits to probabilities
    y_pred_labels = torch.argmax(y_pred_prob, dim=1)  # Get predicted labels

for i in range(num_images):
    plt.subplot(num_rows, 2*num_cols, 2*i+1)
    plot_image(i, y_pred_prob[i].cpu(), y_batch.cpu(), X_batch.squeeze().cpu())
    plt.subplot(num_rows, 2*num_cols, 2*i+2)
    plot_value_array(i, y_pred_prob[i].cpu(), y_batch.cpu())
plt.tight_layout()
plt.show()

# Print Confusion Matrix

In [ ]:
# Try to import torchmetrics and install if it doesn't work
try:
    import torchmetrics
except:
    print("[INFO] coudln't find torchmetrics, installing it...")
    !pip install -q torchmetrics
    import torchmetrics
    print("[INFO] Done!")

In [ ]:
import matplotlib.pyplot as plt
from torchmetrics import ConfusionMatrix
import seaborn as sn
import pandas as pd

# Assuming 'preds' and 'target' are your accumulated tensors of predictions and true labels
# preds should be an integer tensor of predicted labels (e.g., from torch.argmax(outputs, dim=1))
# target should be an integer tensor of true labels

num_classes = 10 # Set the number of classes
metric = ConfusionMatrix(task="multiclass", num_classes=num_classes)

model.eval() # Set model to evaluation mode
all_preds = []
all_targets = []
with torch.no_grad():
    for X, y in test_dataloader:
        X, y = X.to(device), y.to(device)
        pred_logits = model(X)
        pred_labels = pred_logits.argmax(dim=1)
        all_preds.append(pred_labels)
        all_targets.append(y)

# Concatenate all predictions and targets
y_pred = torch.cat(all_preds)
y_true = torch.cat(all_targets)

confmat = ConfusionMatrix(num_classes=num_classes, task='multiclass')
# Ensure tensors are on CPU for torchmetrics if device is CUDA
cm_tensor = confmat(y_pred.cpu(), y_true.cpu())

# Convert to pandas DataFrame for better visualization with seaborn
df_cm = pd.DataFrame(cm_tensor.numpy(), index=range(num_classes), columns=range(num_classes))
plt.figure(figsize = (7,5))
# Use seaborn's heatmap for a nice plot
sn.heatmap(df_cm, annot=True, fmt="d", cmap='Blues')
plt.xlabel("Predicted labels")
plt.ylabel("True labels")
plt.show()

In [ ]:
# check the classes
class_names = train_data.classes
class_names

In [ ]:
import torchmetrics
# Initialize metrics
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
precision = torchmetrics.Precision(task="multiclass", num_classes=num_classes, average='macro')
recall = torchmetrics.Recall(task="multiclass", num_classes=num_classes, average='macro')

# Compute metrics
print(f"Accuracy: {accuracy(y_pred.cpu(), y_true.cpu())}")
print(f"Precision: {precision(y_pred.cpu(), y_true.cpu())}")
print(f"Recall: {recall(y_pred.cpu(), y_true.cpu())}")

# Save and load a model

Let's finish this section off by saving and loading in our best performing model.</p>
<p>We can save and load a PyTorch model using a combination of:</p>
<ul>
<li><code>torch.save</code> - a function to save a whole PyTorch model or a model's <code>state_dict()</code>.</li>
<li><code>torch.load</code> - a function to load in a saved PyTorch object.</li>
<li><code>torch.nn.Module.load_state_dict()</code> - a function to load a saved <code>state_dict()</code> into an existing model instance.</li>
</ul>
<p>You can see more of these three in the <a href="https://pytorch.org/tutorials/beginner/saving_loading_models.html">PyTorch saving and loading models documentation</a>.</p>
<p>For now, let's save our <code>model</code>'s <code>state_dict()</code> then load it back in and evaluate it to make sure the save and load went correctly.</p>

In [ ]:
torch.save(model.state_dict(), "NNmodel_001.pth")
print("Saved PyTorch Model State to NNmodel_001.pth")
# check the size of the file

In [ ]:
NNmodel = NeuralNetwork()
NNmodel.load_state_dict(torch.load("NNmodel_001.pth"))

In [ ]:
labels_map={
0: 'T-shirt',
1: 'Trouser',
2: 'Pullover',
3: 'Dress',
4: 'Coat',
5: 'Sandal',
6: 'Shirt',
7: 'Sneaker',
8: 'Bag',
9: 'Ankle Boot',}

NNmodel.eval()
x, y = test_data[9] # x = data point, Y = label
#x, y = test_data[9][0], test_data[9][1] # [data point][feature, 0], [data point][label, 1]
with torch.no_grad():
    pred = NNmodel(x)
    pred_y= torch.randint((pred[0].argmax(0)), size = (1,)).item()
    y= torch.randint(y, size = (1,)).item()
    predicted, actual = labels_map[pred_y], labels_map[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

In [ ]:
import matplotlib.pyplot as plt
image = x
label = y
# image, label = test_data[9]
print(f"Image shape: {image.shape}")
plt.imshow(image.squeeze()) # image shape is [1, 28, 28] (colour channels, height, width)
plt.title(class_names[label])
plt.show()

# Deployment on Raspberry Pi

In [ ]:
!pip install onnx

In [ ]:
# Install onnxscript as it's a dependency for torch.onnx.export in newer PyTorch versions
!pip install onnxscript

In [ ]:
## Convert the PyTorch model to ONNX
import torch
import torch.onnx

NNmodel.eval()

# Define dummy input (match your input size)
dummy_input = torch.randn(1, 28, 28).to(device)

# Convert to ONNX format
# check the size of the output file
torch.onnx.export(NNmodel, dummy_input, "model.onnx", export_params=True, opset_version=11,
                  input_names=['input'], output_names=['output'])

In [ ]:
## Download some test images to run rpi

import torch
import matplotlib.pyplot as plt
import os
from torchvision.utils import save_image

# Ensure the test_data dataset is available before running this script
torch.manual_seed(42)
fig = plt.figure(figsize=(9, 9))
rows, cols = 4, 4

# Create a directory to save sample images
save_dir = "sample_images"
os.makedirs(save_dir, exist_ok=True)

for i in range(1, rows * cols + 1):
    random_idx = torch.randint(0, len(test_data), size=[1]).item()
    img, label = test_data[random_idx]

    # Convert image to tensor (if it's a PIL image)
    if isinstance(img, torch.Tensor):
        img_tensor = img
    else:
        img_tensor = transforms.ToTensor()(img)

    # Save the image
    save_path = os.path.join(save_dir, f"{i}_{class_names[label]}.png")
    save_image(img_tensor, save_path)

    # Plot the image
    ax = fig.add_subplot(rows, cols, i)
    ax.imshow(img_tensor.squeeze(), cmap="gray")
    ax.set_title(f"Label: {class_names[label]}")
    ax.axis("off")

plt.show()

In [ ]:
!zip -r sample_images.zip sample_images

Transfer these images to RPi

## Instructions for RPi

<strong> Install ONNX Runtime</strong>
1. Download the source to your Raspberry Pi
<code>  git clone https://github.com/cassiebreviu/onnxruntime-raspberrypi.git
</code>
2. Navigate to the onnxruntime-raspberrypi download location and install the package from the requirements.txt with the following command.
<p><code>cd onnxruntime-raspberrypi</code></p>
<p><code>pip install -r requirements.txt --break-system-packages</code></p>